# Ticket Classifier

This notebook audits the raw CSV first, then builds a corrected category
classifier from consistent ticket examples. The original CSV labels are not
safe to train on directly because the same ticket text appears under many
different categories.


In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.pipeline import Pipeline


In [ ]:
raw_df = pd.read_csv("customer_support_tickets_200k.csv")
raw_df = raw_df[["issue_description", "category", "priority"]].dropna().copy()

print("Rows:", len(raw_df))
print("Unique issue descriptions:", raw_df["issue_description"].nunique())
print("Unique raw categories:", raw_df["category"].nunique())

raw_df.head()


In [ ]:
issue_audit = (
    raw_df.groupby("issue_description")
    .agg(
        rows=("category", "size"),
        unique_categories=("category", "nunique"),
    )
    .sort_values("rows", ascending=False)
)

print(issue_audit)

contradicted_rows = int((issue_audit["unique_categories"] > 1).sum())
print()
print("Descriptions with contradictory labels:", contradicted_rows)

issue_audit


In [ ]:
for text in sorted(raw_df["issue_description"].unique()):
    categories = sorted(raw_df.loc[raw_df["issue_description"] == text, "category"].unique())
    print(text)
    print("Raw categories:", ", ".join(categories))
    print()


## Corrected Training Data

The raw file only contains 10 repeated ticket texts, and all 10 are mapped to
conflicting categories. Because of that, we do not train on the raw `category`
column.

Instead, we build a compact corrected dataset from the meaning of the ticket
text itself. Two raw categories, `Account Suspension` and `Feature Request`,
are not supported by the available descriptions, so they are excluded from the
corrected model.


In [ ]:
training_examples = [
    ("I am experiencing very slow performance while using the dashboard.", "Performance Issue"),
    ("The dashboard is taking too long to respond.", "Performance Issue"),
    ("The portal feels sluggish whenever I open reports.", "Performance Issue"),
    ("Pages load very slowly in the web app.", "Performance Issue"),
    ("The application performance has become extremely slow.", "Performance Issue"),
    ("Navigating the dashboard is delayed and unresponsive.", "Performance Issue"),

    ("I am unable to access my account after entering the correct credentials.", "Login Issue"),
    ("I cannot log in even though my password is correct.", "Login Issue"),
    ("The system rejects my valid username and password.", "Login Issue"),
    ("My account sign-in fails with the right login details.", "Login Issue"),
    ("I am locked out despite using the correct credentials.", "Login Issue"),
    ("Unable to login into my account.", "Login Issue"),

    ("I found a bug in the latest update affecting report generation.", "Bug Report"),
    ("The application crashes whenever I try to upload a file.", "Bug Report"),
    ("Uploading a file makes the app close unexpectedly.", "Bug Report"),
    ("The latest release introduced a report generation bug.", "Bug Report"),
    ("The app throws an error whenever I attach a document.", "Bug Report"),
    ("A software bug is preventing file uploads from working.", "Bug Report"),

    ("I would like to request a refund for the recent charge.", "Refund Request"),
    ("Please issue a refund for my last payment.", "Refund Request"),
    ("I need my money back for the recent transaction.", "Refund Request"),
    ("I was charged incorrectly and want a refund.", "Refund Request"),
    ("Can you refund the amount taken from my account?", "Refund Request"),
    ("Refund not received after payment failed.", "Refund Request"),

    ("My subscription was cancelled without my request and I need clarification.", "Subscription Cancellation"),
    ("My plan ended even though I did not cancel it.", "Subscription Cancellation"),
    ("The subscription was stopped without my approval.", "Subscription Cancellation"),
    ("Why was my membership cancelled without permission?", "Subscription Cancellation"),
    ("My service subscription disappeared unexpectedly.", "Subscription Cancellation"),
    ("The plan was terminated without my consent.", "Subscription Cancellation"),

    ("The payment was deducted from my bank account but the transaction shows failed.", "Payment Problem"),
    ("There seems to be a discrepancy in my billing statement for this month.", "Payment Problem"),
    ("Money was debited but the payment did not go through.", "Payment Problem"),
    ("I was charged but the purchase still failed.", "Payment Problem"),
    ("My bill contains an incorrect charge this month.", "Payment Problem"),
    ("Payment failed and amount deducted from my bank account.", "Payment Problem"),

    ("The system is not syncing data across devices properly.", "Data Sync Issue"),
    ("My data is not updating between devices.", "Data Sync Issue"),
    ("Information entered on one device does not appear on another.", "Data Sync Issue"),
    ("The app fails to sync records across platforms.", "Data Sync Issue"),
    ("Changes are not syncing between mobile and web.", "Data Sync Issue"),
    ("The account is not syncing changes across devices.", "Data Sync Issue"),

    ("Two-factor authentication codes are not being delivered to my phone.", "Security Concern"),
    ("I am not receiving the OTP code on my mobile.", "Security Concern"),
    ("Verification codes never arrive for two-factor login.", "Security Concern"),
    ("My 2FA messages are not coming through.", "Security Concern"),
    ("Security code delivery is failing during sign in.", "Security Concern"),
    ("I never receive the two-factor authentication text.", "Security Concern"),
]

train_df = pd.DataFrame(training_examples, columns=["text", "category"])

supported_categories = sorted(train_df["category"].unique())
unsupported_categories = sorted(set(raw_df["category"].unique()) - set(supported_categories))

print("Supported corrected categories:", supported_categories)
print("Unsupported raw categories removed from the model:", unsupported_categories)
print("Training rows:", len(train_df))

train_df.head()


In [ ]:
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model.fit(train_df["text"], train_df["category"])

print("Corrected model training completed.")


In [ ]:
holdout_examples = [
    ("Dashboard freezes for several seconds before loading.", "Performance Issue"),
    ("The website responds very slowly when I switch tabs.", "Performance Issue"),

    ("I cannot sign in to my account with the correct password.", "Login Issue"),
    ("Valid credentials still will not let me log in.", "Login Issue"),

    ("Report export broke after the update.", "Bug Report"),
    ("Uploading attachments crashes the application.", "Bug Report"),

    ("I want a refund for the charge on my card.", "Refund Request"),
    ("Please return the payment that was taken from me.", "Refund Request"),

    ("My subscription got cancelled and I never asked for it.", "Subscription Cancellation"),
    ("The plan was shut down without my approval.", "Subscription Cancellation"),

    ("The amount was deducted but checkout still says failed.", "Payment Problem"),
    ("My invoice shows charges that do not look correct.", "Payment Problem"),

    ("Data entered on my phone is not showing on the laptop.", "Data Sync Issue"),
    ("My records are not updating across devices.", "Data Sync Issue"),

    ("OTP codes for login are not reaching my phone.", "Security Concern"),
    ("I never receive the security verification message.", "Security Concern"),
]

holdout_df = pd.DataFrame(holdout_examples, columns=["text", "expected_category"])
holdout_df["predicted_category"] = model.predict(holdout_df["text"])

accuracy = accuracy_score(holdout_df["expected_category"], holdout_df["predicted_category"])
print("Holdout accuracy:", round(accuracy, 4))
print()
print(classification_report(holdout_df["expected_category"], holdout_df["predicted_category"]))

holdout_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ConfusionMatrixDisplay.from_predictions(
    holdout_df["expected_category"],
    holdout_df["predicted_category"],
    xticks_rotation=45,
    ax=ax,
    colorbar=False,
)
plt.title("Confusion Matrix on Curated Holdout Set")
plt.tight_layout()
plt.show()


## Sample Predictions

These examples use the corrected model, not the corrupted raw labels.


In [ ]:
sample = ["Payment failed and amount deducted from my bank account"]
prediction = model.predict(sample)[0]
print("Predicted Category:", prediction)


In [ ]:
sample = ["Unable to login into my account"]
prediction = model.predict(sample)[0]
print("Predicted Category:", prediction)


In [ ]:
sample = ["Refund not received after payment failed"]
prediction = model.predict(sample)[0]
print("Predicted Category:", prediction)
